In [1]:
# download and install ollama on a local pc (safety-your data will not leave the pc)

In [ ]:
# pipenv install langgraph 
# pipenv install langchain-core 
# pipenv install langchain-ollama


# optional:
# pipenv install python-dotenv 

In [3]:
from typing import Annotated, Sequence
from typing_extensions import TypedDict

# Importy pro LangGraph a zprávy
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage

import pandas as pd
import numpy as np

# Data

In [4]:
# Simulating a real order table (Order ID, product, price per unit, quantity)
orders_data = {
    "order_id": [101, 102, 103, 104, 105],
    "product": ["Notebook", "Monitor", "Notebook", "Mouse", "Keyboard"],
    "price_per_unit": [1000.0, 200.0, 1000.0, 50.0, 100.0],
    "quantity": [1, 2, 1, 4, 2]
}
df = pd.DataFrame(orders_data)

# Calculate total row price (price * quantity)
df["total_row_price"] = df["price_per_unit"] * df["quantity"]
df

,order_id,product,price_per_unit,quantity,total_row_price
0,101,Notebook,1000.0,1,1000.0
1,102,Monitor,200.0,2,400.0
2,103,Notebook,1000.0,1,1000.0
3,104,Mouse,50.0,4,200.0
4,105,Keyboard,100.0,2,200.0


# Function

In [5]:
# 1. Tool to calculate TOTAL REVENUE (sum of all orders)
def get_total_revenue() -> float:
    print("🔧 [Tool] Calculating total revenue from DataFrame...")
    total_revenue = df["total_row_price"].sum()
    print(f"   => Calculated: {total_revenue} USD")
    return float(total_revenue)


# 2. Tool to calculate AVERAGE PRICE (mean of unique product prices)
def get_average_price() -> float:
    print("🔧 [Tool] Calculating average product price from DataFrame...")
    average_price = df["price_per_unit"].mean()
    print(f"   => Calculated: {average_price} USD")
    return float(average_price)

# Tools

In [6]:
# Tools definition for the LLM (Ollama needs to know the structure)
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_total_revenue",
            "description": "Returns the total revenue calculated from all store orders."
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_average_price",
            "description": "Returns the average price of a single product from the catalog."
        }
    }
]


# Brain of the Agent = State

In [7]:
from typing import TypedDict, Annotated
import operator
from langchain_ollama import ChatOllama 
from langchain_core.messages import BaseMessage

# LANGGRAPH STATE A OLLAMA

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

# Initialize ChatOllama with the Qwen model for reliable tool calling
llm = ChatOllama(model="qwen2.5", 
                 temperature=0.0,
                 #num_predict=4096
                 )

# Bind the Pandas analytical tools to the model
llm_with_tools = llm.bind_tools(TOOLS)

def node_agent(state: AgentState):
    print("\n🤖 [Node: Agent] Model is thinking...")
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


In [8]:
from langchain_core.messages import ToolMessage

# UZEL PRO SPUŠTĚNÍ NÁSTROJŮ

def node_tools(state: AgentState):
    print("\n🛠️ [Node: Tools] Executing requested functions...")
    
    # Get the last message from the history sent by the agent node
    last_message = state["messages"][-1]
    tool_outputs = []
    
    # Iterate through all tool calls requested by the model
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_call_id = tool_call["id"]
        
        # Execute the real Python/Pandas function based on its name
        if tool_name == "get_total_revenue":
            result = get_total_revenue()
        elif tool_name == "get_average_price":
            result = get_average_price()
        else:
            result = f"Error: Tool '{tool_name}' not found."
            
        # Wrap the result into a ToolMessage required by LangGraph
        tool_outputs.append(
            ToolMessage(
                content=str(result),
                name=tool_name,
                tool_call_id=tool_call_id
            )
        )
        
    return {"messages": tool_outputs}


# GRAPH COMPOSITION AND EXECUTION

In [13]:
from langgraph.graph import StateGraph, START, END


# 1. Initialize the graph with our custom state (AgentState)
workflow = StateGraph(AgentState)

# 2. Add our nodes to the graph
workflow.add_node("agent", node_agent)
workflow.add_node("tools", node_tools)

# 3. Set the entry point of the graph (always start with the brain/agent)
workflow.add_edge(START, "agent")

# 4. Define the routing function (Conditional Edge)
# Checks after each agent step if the model wants to call tools or finish
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    
    # If the model requested tool calls, route to the "tools" node
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    
    # Otherwise, stop the graph and return the final answer to the user
    return END

# Connect the "agent" node to the conditional routing function
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",  # If should_continue returns "tools", go to tools node
        END: END           # If should_continue returns END, finish the graph
    }
)

# 5. The "tools" node must return control back to the agent to evaluate results
workflow.add_edge("tools", "agent")

# 6. Compile the workflow into a runnable graph application
app = workflow.compile()


# Test

In [10]:
# FUNKCE PRO SPUŠTĚNÍ JEDNOHO DOTAZU

def spust_agenta(dotaz):
    print(f"👤 Uživatel píše: '{dotaz}'")

    # Spustíme graf a předáme mu úvodní zprávu od uživatele
    final_state = app.invoke({"messages": [HumanMessage(content=dotaz)]})

    print("\n========================================================")
    print("🎉 GRAF SKONČIL. ZDE JE FINÁLNÍ ODPOVĚĎ:")
    # Vytáhneme úplně poslední zprávu, kterou graf vyprodukoval
    print(final_state["messages"][-1].content)




In [11]:
# JEDNODUCHÉ SPUŠTĚNÍ

spust_agenta("Ahoj! Spočítej mi prosím, jaký je rozdíl mezi celkovými tržbami a průměrnou cenou produktu.")

👤 Uživatel píše: 'Ahoj! Spočítej mi prosím, jaký je rozdíl mezi celkovými tržbami a průměrnou cenou produktu.'

🤖 [Node: Agent] Model is thinking...

🛠️ [Node: Tools] Executing requested functions...
🔧 [Tool] Calculating total revenue from DataFrame...
   => Calculated: 2800.0 USD

🤖 [Node: Agent] Model is thinking...

🛠️ [Node: Tools] Executing requested functions...
🔧 [Tool] Calculating average product price from DataFrame...
   => Calculated: 470.0 USD

🤖 [Node: Agent] Model is thinking...

🎉 GRAF SKONČIL. ZDE JE FINÁLNÍ ODPOVĚĎ:
Dostali jsme informaci, že průměrná cena produktu je 470 Kč.

Teď mohu vypočítat rozdíl mezi celkovými tržbami a průměrnou cenou produktu. Celkové tržby činily 2800 Kč, zatímco průměrná cena produktu byla 470 Kč.

Rozdíl mezi těmito hodnotami je: 2800 - 470 = 2330 Kč.
